In [ ]:
# Notebook: Interactive Low-Pass Filter Analysis with Legends Outside Top-Right
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def plot_interactive_low_pass(omega_c):
    omega_vals = np.linspace(-25, 25, 1200)
    
    # 1. Ideal Low-Pass Magnitude Response
    ideal_lp = np.where(np.abs(omega_vals) <= omega_c, 1.0, 0.0)
    
    # 2. Real Low-Pass Magnitude Response with smoother blending between regions
    real_lp = np.zeros_like(omega_vals)
    transition_width = 3.5
    
    for i, w in enumerate(np.abs(omega_vals)):
        if w <= omega_c:
            # Passband with minor ripples, smoothly fading near omega_c
            fade = 1.0 - 0.2 * max(0.0, (w - 0.7 * omega_c) / (0.3 * omega_c)) if omega_c > 0 else 1.0
            real_lp[i] = 1.0 + (0.04 * np.cos(3 * w) * fade)
        elif omega_c < w <= omega_c + transition_width:
            # Smooth transition band bridging 1.0 down towards 0 using a cosine taper
            fraction = (w - omega_c) / transition_width
            # Smooth cosine interpolation from ~1.0 to 0.0 passing through 0.707
            real_lp[i] = 0.5 * (1.0 + np.cos(np.pi * fraction)) + 0.05 * np.exp(-fraction)
        else:
            # Stopband with small attenuation ripples, starting smoothly from 0
            dist = w - (omega_c + transition_width)
            attenuation_factor = 1.0 - np.exp(-dist)
            real_lp[i] = (0.04 * np.sin(2 * w)) * attenuation_factor

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    
    # Plot Ideal Low-Pass Filter
    ax1.plot(omega_vals, ideal_lp, 'b-', linewidth=2.5, label=r"Ideal LP $|\mathcal{H}_{\ell p}(j\omega)|$")
    ax1.axvline(omega_c, color='red', linestyle='--', linewidth=1.5, label=r"Cutoff $\omega_c = %.1f$" % omega_c)
    ax1.axvline(-omega_c, color='red', linestyle='--', linewidth=1.5)
    ax1.set_title(r"Interactive Ideal and Real Low-Pass Filter Responses", fontsize=13)
    ax1.set_ylabel(r"Amplitude $|\mathcal{H}(j\omega)|$", fontsize=11)
    ax1.grid(True, linestyle=":", alpha=0.7)
    ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax1.set_ylim(-0.15, 1.25)
    # Legend placed outside upper right
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)
    
    # Plot Real Low-Pass Filter
    ax2.plot(omega_vals, real_lp, 'g-', linewidth=2, label=r"Real LP (ripples & transition)")
    ax2.axhline(0.7071, color='orange', linestyle=':', linewidth=1.5, label=r"-3 dB point ($0.7071$)")
    ax2.axvline(omega_c, color='red', linestyle='--', linewidth=1.5, label=r"Cutoff $\omega_c$")
    ax2.axvline(-omega_c, color='red', linestyle='--', linewidth=1.5)
    ax2.set_xlabel(r"Frequency $\omega$ (rad/s)", fontsize=11)
    ax2.set_ylabel(r"Amplitude $|\mathcal{H}(j\omega)|$", fontsize=11)
    ax2.grid(True, linestyle=":", alpha=0.7)
    ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax2.set_ylim(-0.15, 1.25)
    # Legend placed outside upper right
    ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Interactive widget slider
widgets.interact(
    plot_interactive_low_pass, 
    omega_c=widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5, description=r'$\omega_c$ (rad/s):', style={'description_width': 'initial'})
);